In [1]:
# ============================================================================
# ENTERPRISE HR CHATBOT - FAST & CLEAN EDITION
# ============================================================================

import os
import sys
import logging
import warnings
import textwrap
from dotenv import load_dotenv
from pydantic import BaseModel, Field

# 1. SUPPRESS UGLY WARNINGS
# ----------------------------------------------------------------------------
warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("transformers").setLevel(logging.ERROR)

# LangChain & AI Libraries
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from openai import OpenAI as OpenAIClient

# Load environment variables
load_dotenv()

True

In [2]:
# ============================================================================
# 2. CONFIGURATION (SPEED OPTIMIZED)
# ============================================================================
class SystemConfig(BaseModel):
    file_path: str = Field(default="Egyptian_HR_Policy_Manual_Complete.pdf", description="Path to PDF")
    
    # AI SETTINGS
    embedding_model: str = "sentence-transformers/all-MiniLM-L6-v2"
    
    # SPEED HACK: Smaller chunks = Faster LLM processing
    # We keep k=10 to find the page, but read less text per page.
    chunk_size: int = 600 
    chunk_overlap: int = 100
    
    # LM STUDIO SETTINGS
    lm_studio_base_url: str = "http://localhost:1234/v1"
    lm_studio_api_key: str = "lm-studio"
    lm_studio_model: str = "llama-3.2-3b-instruct"
    temperature: float = 0.3
    
    # RETRIEVAL SETTINGS
    similarity_k: int = 10 

    class Config:
        validate_assignment = True

In [3]:
# ============================================================================
# 3. LOGGING SETUP
# ============================================================================
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(message)s',
    handlers=[
        logging.FileHandler('hr_chatbot.log', encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

In [4]:
# ============================================================================
# 4. LLM CLIENT (LM STUDIO ADAPTER)
# ============================================================================
class LocalLLM:
    def __init__(self, base_url: str, api_key: str, model: str, temperature: float = 0.7):
        self.client = OpenAIClient(base_url=base_url, api_key=api_key)
        self.model = model
        self.temperature = temperature

    def predict(self, prompt: str) -> str:
        messages = [
            {"role": "system", "content": "You are a professional HR Assistant. Format your answers clearly using Markdown."},
            {"role": "user", "content": prompt}
        ]
        try:
            completion = self.client.chat.completions.create(
                model=self.model,
                messages=messages,
                temperature=self.temperature,
                timeout=90.0
            )
            return completion.choices[0].message.content or ""
        except Exception as e:
            return f"Error connecting to LM Studio: {e}"


In [5]:

# ============================================================================
# 5. MAIN CHATBOT CLASS
# ============================================================================
class EnterpriseHRChatbot:
    def __init__(self, config: SystemConfig):
        self.config = config
        self.vector_db = None
        self.embeddings = None
        self.llm = None
        
        logger.info("--- INITIALIZING HR CHATBOT (Final Edition) ---")
        self._ingest_documents()
        self._init_model()
        logger.info("--- READY ---")

    def _ingest_documents(self):
        try:
            if not os.path.exists(self.config.file_path):
                raise FileNotFoundError(f"PDF not found: {self.config.file_path}")

            logger.info(f"Loading PDF: {self.config.file_path}...")
            loader = PyPDFLoader(self.config.file_path)
            docs = loader.load()
            
            # Filter tiny pages
            docs = [d for d in docs if len(d.page_content) > 50]
            
            splitter = RecursiveCharacterTextSplitter(
                chunk_size=self.config.chunk_size, 
                chunk_overlap=self.config.chunk_overlap
            )
            chunks = splitter.split_documents(docs)
            
            logger.info("Building Vector Index...")
            self.embeddings = HuggingFaceEmbeddings(model_name=self.config.embedding_model)
            self.vector_db = FAISS.from_documents(chunks, self.embeddings)
            
        except Exception as e:
            logger.error(f"Ingestion failed: {e}")
            sys.exit(1)

    def _init_model(self):
        self.llm = LocalLLM(
            base_url=self.config.lm_studio_base_url,
            api_key=self.config.lm_studio_api_key,
            model=self.config.lm_studio_model,
            temperature=self.config.temperature
        )

    def query(self, question: str):
        # 1. Retrieve (Get top 10 chunks)
        docs = self.vector_db.similarity_search(question, k=self.config.similarity_k)
        
        # Build Context String (Removed Page Numbers from Prompt to save tokens)
        context = "\n".join([d.page_content for d in docs])
        
        # 2. Prompt with Formatting Instructions
        prompt = f"""You are a helpful HR Assistant. Answer the question based strictly on the context below.

FORMATTING RULES:
1. Start with a direct answer.
2. Use **Bold Headers** for key sections.
3. Use bullet points (•) for lists.
4. Keep paragraphs short.
5. If the specific term isn't found, look for synonyms like "Commuted Leave", "Medical Leave" or "Half Pay Leave".

CONTEXT:
{context}

QUESTION: 
{question}

ANSWER:"""
        
        # 3. Generate
        answer = self.llm.predict(prompt)
        
        return answer

In [22]:

# ============================================================================
# 6. BEAUTIFUL OUTPUT PRINTER (SOURCES REMOVED)
# ============================================================================
def print_boxed(title, text):
    width = 135 # Adjusted width for readability
    
    # Border characters
    horizontal = "═" * (width - 2)
    top_border = "╔" + horizontal + "╗"
    bottom_border = "╚" + horizontal + "╝"
    divider = "╠" + horizontal + "╣"
    
    print("\n" + top_border)
    print(f"║ {title.center(width - 4)} ║")
    print(divider)
    
    wrapper = textwrap.TextWrapper(
        width=width-6, 
        initial_indent="  ", 
        subsequent_indent="    ",
        break_long_words=False,
        replace_whitespace=False
    )
    
    for line in text.split('\n'):
        clean_line = line.strip()
        if clean_line.startswith('**'):
             print(f"║  {clean_line:<{width-6}}  ║")
        elif clean_line.startswith('-') or clean_line.startswith('•') or clean_line.startswith('*'):
             wrapped_bullets = wrapper.wrap(clean_line)
             for i, w_line in enumerate(wrapped_bullets):
                 if i == 0:
                     print(f"║ {w_line:<{width-4}} ║")
                 else:
                     print(f"║     {w_line.strip():<{width-8}} ║")
        elif clean_line == "":
             print(f"║ {' ':<{width-4}} ║")
        else:
            for wrapped_line in wrapper.wrap(line):
                print(f"║ {wrapped_line:<{width-4}} ║")
    
    print(bottom_border + "\n")


In [7]:
pip install --upgrade openai

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [19]:
pip install arabic-reshaper python-bidi

Defaulting to user installation because normal site-packages is not writeable

   ---------------------------------------- 0/2 [python-bidi]
   -------------------- ------------------- 1/2 [arabic-reshaper]
   ---------------------------------------- 2/2 [arabic-reshaper]

Note: you may need to restart the kernel to use updated packages.


In [23]:
# ============================================================================
# 7. RUNNER
# ============================================================================
if __name__ == "__main__":
    bot = EnterpriseHRChatbot(SystemConfig())
    
    questions = [
        "What is the policy on sick leave?",
        "How many days of casual leave am I entitled to?",
        "What are the rules for maternity leave?",
        "Is there a provident fund scheme?",
        "What is the procedure for resignation?"
    ]
    
    print("\n" + "="*80)
    print(f"STARTING BATCH TEST ({len(questions)} Questions)")
    print("="*80)

    for i, q in enumerate(questions, 1):
        print(f"\n QUESTION {i}: {q}")
        
        # Query the bot
        answer = bot.query(q)
        
        # Print Beautiful Result (No Sources)
        print_boxed(f"ANSWER {i}", answer)

2026-02-18 23:05:51,312 - --- INITIALIZING HR CHATBOT (Final Edition) ---
2026-02-18 23:05:51,315 - Loading PDF: Egyptian_HR_Policy_Manual_Complete.pdf...
2026-02-18 23:06:20,801 - Building Vector Index...
2026-02-18 23:07:38,552 - --- READY ---



STARTING BATCH TEST (5 Questions)

 QUESTION 1: What is the policy on sick leave?


2026-02-18 23:07:55,101 - HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"



╔═════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╗
║                                                               ANSWER 1                                                              ║
╠═════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╣
║  **Sick Leave Policy**                                                                                                              ║
║                                                                                                                                     ║
║   The company's sick leave policy allows employees to take up to 180 days of sick leave per calendar year, which can be taken as    ║
║     separate periods or continuously. The compensation during sick leave varies based on the duration of absence.                   ║
║                                              

2026-02-18 23:08:06,309 - HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"


...


╔═════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╗
║                                                               ANSWER 3                                                              ║
╠═════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╣
║  **Maternity Leave Rules**                                                                                                          ║
║                                                                                                                                     ║
║   The following are the rules for maternity leave:                                                                                  ║
║                                                                                                                                     ║
║   * **Duration**: Maximum of 120 days (increa

2026-02-18 23:08:26,967 - HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"



╔═════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╗
║                                                               ANSWER 4                                                              ║
╠═════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╣
║  **Yes**                                                                                                                            ║
║                                                                                                                                     ║
║   The following information supports the existence of a provident fund scheme:                                                      ║
║                                                                                                                                     ║
║   • **Payslip Requirements**: The payslip mus

2026-02-18 23:08:33,851 - HTTP Request: POST http://localhost:1234/v1/chat/completions "HTTP/1.1 200 OK"



╔═════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╗
║                                                               ANSWER 5                                                              ║
╠═════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╣
║  **Resignation Procedure**                                                                                                          ║
║   To resign from your position, you must follow these steps:                                                                        ║
║                                                                                                                                     ║
║   • Submit a written resignation letter to your supervisor.                                                                         ║
║   • Provide the required notice period as spe